# Lecture 10. A practice case: choosing what to model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture10/lecture10.ipynb)

Applied Quantitative Methods, Fall 2026. Readings: all the readings of Lectures 6 to 9.

The first half of the lecture answers your questions and goes through the oral exam; it has no cells here. This
notebook follows the second half, a practice case on new data. In Lectures 6 to 9 the slides chose the outcome and
the explanatory variables for you. Today we practice making those choices:

| Part | What we do |
|---|---|
| 3 | Questions to ask about an outcome and about each explanatory variable; the case and the data |
| 4 | Which contacts are escalated: two similar logit models, one contact by hand, a grid of contacts, a threshold |
| 5 | What the results can and cannot say, and a summary for management |

Part 4 ends in two exercises: you predict a result, run the cell, and change one value. You do not need to write any
code. The workshop uses the same data for two other questions, and there you make the choices yourselves.

Run the setup cell first. It loads the data from the course repository.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
contacts = pd.read_csv(url + "service_contacts.csv")   # 2,000 contacts from one period

print(f"Contacts: {len(contacts):,}")
contacts.head()

## Part 3. Choosing what to model

Before anything is fitted, three choices have to be made: the outcome, the explanatory variables, and, when two
models are close, which one to keep.

**Questions to ask about the outcome**

| Question | Why it matters |
|---|---|
| Does the column measure what management asks about? | A model of another column answers another question |
| Is it a number or yes/no? | A number: linear regression. Yes/no: the LPM, logit or probit |
| Is it skewed, with a long right tail? | Logs turn coefficients into percent differences and often even out the residuals |
| Is it there for every row? | Missing values shrink the sample, and they may be missing for a reason |

**Questions to ask about each explanatory variable**

| Question | If not, what goes wrong |
|---|---|
| Is it known before the outcome happens? | It may be a result of the outcome, so it explains it backwards |
| Is it something other than a part of the outcome? | A part explains the whole by construction: R² near 1, and nothing learned |
| Does it measure something other than an x already in the model? | Two near copies share the information: large standard errors, high VIF |
| Is it more than a label? | An ID or a running number means nothing as a number |
| Does the question need it? | A control left out can bias the coefficient you care about; a variable with no link adds noise |

**Comparing two similar models.** Fit both on the same rows. Look at the coefficient you care about (does it move?),
at the added x (t or z and p; for logit also the likelihood ratio test), and at the fit (adjusted R² for linear
regression; pseudo R² and the log-likelihood for logit). A higher R² alone is no reason to keep a variable: a column
that is part of the outcome raises R² the most of all.

**An earlier value of the outcome.** When the data hold the outcome measured earlier, before the thing you study,
adding it to the model compares units that started at the same level:

ln y_after = β0 + β1·course + β2·ln y_before + ... + ε

β1 is then the difference after, between units with the same value before. When β2 is near 1, that is close to a
difference in change.
If β1 moves when the earlier value is added, the groups differed before. Compare the two models on the same rows:
units without an earlier value drop out, and the results describe the units that remain. With logs on both sides, β2
is an elasticity (Lecture 7).

### TeleNag's customer service center

TeleNag sells internet and mobile subscriptions. Its customer service center handles contacts by phone and
chat. Some agents have taken a course on handling and documenting contacts; managers chose whom to send. The file
holds 2,000 contacts drawn at random from all contacts this autumn; the center has several sites and many hundreds of
agents, so most agents appear only once or twice. The company is invented and the data are simulated for the course. Management has three questions:

1. What goes with how much agent time a contact takes, and is the course associated with less agent time? (Workshop)
2. Which contacts are followed by the customer contacting the center again within seven days? (Workshop)
3. Which contacts end up with the second-line team, and when should a specialist be ready? (Today)

The columns are described in `data/README.md` in the repository and on the slide "The data". Look at them below.

In [ ]:
contacts.info()

In [ ]:
contacts.describe().round(2)

## Part 4. Escalation in different scenarios

**Question.** Which contacts end up with the second-line team, and when should a specialist be ready?

**The outcome.** `escalated` is 1 if the agent passed the issue on to the second-line team at the end of the contact.
It is yes or no, so we use logit.

**The x's.** Candidates are the columns known when the contact starts. Today we keep the model small: issue type,
the agent's experience and the customer segment. `channel`, `agent_trained`, `agent_minutes_before`,
`queue_minutes`, `shift` and `customer_years` are also known at the start; they are left for the workshop. That is a simplification: adding `agent_trained`, for example, moves the experience coefficient a little (from −0.0148 to
−0.0167). `transfers` and `handling_minutes` happen during the contact, on the way to an escalation;
`wrapup_minutes`, `total_minutes`, `repeat_contact_7d` and `satisfaction` are measured after it. `contact_id` is a
label, and `agent_contacts_handled` is a near copy of experience. None of these belong in a model of escalation.

Experience enters in months, which keeps the calculation by hand simple. The log of experience fits slightly better
here (log-likelihood −707.46 against −709.50 for model B below).

In [ ]:
print(f"Escalated: {contacts['escalated'].sum()} of {len(contacts):,} ({contacts['escalated'].mean():.3f})")
print(contacts.groupby("issue_type")["escalated"].mean().round(3))

### Two similar models

Model A has issue type and experience. Model B adds segment. Is segment worth keeping? Both models use all 2,000
rows, since none of the columns has missing values.

In [ ]:
model_a = smf.logit("escalated ~ C(issue_type) + agent_experience_months", data=contacts).fit()
model_b = smf.logit("escalated ~ C(issue_type) + agent_experience_months + C(segment)", data=contacts).fit()

LR = 2 * (model_b.llf - model_a.llf)
print(f"Log-likelihood A {model_a.llf:.3f}, B {model_b.llf:.3f}")
print(f"LR = {LR:.2f}, p = {stats.chi2.sf(LR, 1):.4f}")
print(f"Pseudo R2 A {model_a.prsquared:.3f}, B {model_b.prsquared:.3f}")
print(f"Experience coefficient A {model_a.params['agent_experience_months']:.4f}, "
      f"B {model_b.params['agent_experience_months']:.4f}")

In [ ]:
print(model_b.summary())

Segment is significant, and the likelihood ratio test agrees. The experience coefficient hardly moves, so segment does
not explain the experience association. We keep model B. The reference categories are Billing and Business, the first in the
alphabet. As in Lecture 9, the coefficients give the sign and the significance; the sizes come as probabilities.

### One contact by hand

A business customer with a technical issue, handled by an agent with 12 months in the job. Business is the reference,
so only the Technical coefficient and the experience term are added to the intercept. Then the same contact with an
agent who has 96 months in the job.

In [ ]:
b = model_b.params
for months in [12, 96]:
    z = b["Intercept"] + b["C(issue_type)[T.Technical]"] + b["agent_experience_months"] * months
    p = 1 / (1 + np.exp(-z))
    print(f"{months:>3} months: z = {z:.3f}, p = {p:.3f}")

### A grid of contacts

One row per scenario. Every x in the formula needs a column; a single value fills every row.

In [ ]:
grid = pd.DataFrame({"issue_type": "Technical", "segment": "Business",
                     "agent_experience_months": [3, 12, 36, 96]})
grid["p"] = model_b.predict(grid)
print(grid[["agent_experience_months", "p"]].round(3))

### Exercise 1

For a private customer's billing issue, is the difference between 3 and 96 months larger or smaller than for a
business customer's technical issue (26.3 points; the notebook computes differences before rounding)? Write your prediction, run the cell, then change `issue` to
`"Billing"` and `segment` to `"Private"` and run it again.

In [ ]:
issue = "Technical"      # change to "Billing"
segment = "Business"     # change to "Private"

probabilities = []
for months in [3, 12, 36, 96]:
    contact = pd.DataFrame({"issue_type": [issue], "segment": [segment], "agent_experience_months": [months]})
    p = model_b.predict(contact).iloc[0]
    probabilities.append(p)
    print(f"{months:>3} months: {p:.3f}")

print(f"Difference, 3 against 96 months: {100 * (probabilities[0] - probabilities[-1]):.1f} percentage points")

The linear probability model, the same model fitted by OLS, gives the same difference for every contact, and a
probability below 0 for the billing contact at 96 months:

In [ ]:
lpm = smf.ols("escalated ~ C(issue_type) + agent_experience_months + C(segment)", data=contacts).fit(cov_type="HC1")
for issue, segment in [("Technical", "Business"), ("Billing", "Private")]:
    contact = pd.DataFrame({"issue_type": issue, "segment": segment, "agent_experience_months": [3, 96]})
    p = lpm.predict(contact)
    print(f"{issue}, {segment}: {p.iloc[0]:.3f} and {p.iloc[1]:.3f}, "
          f"difference {100 * (p.iloc[0] - p.iloc[1]):.1f} percentage points")

### The same experience, a different difference

The experience coefficient is the same for every contact, but it moves the index z, and how much that moves the
probability depends on where the contact is on the curve. Experience goes with the largest drop in escalations for
technical issues from business customers, and changes little for billing issues.

In [ ]:
months = np.arange(0, 121, 2)
for issue, segment, style in [("Technical", "Business", "-"), ("Technical", "Private", "--"),
                              ("Billing", "Business", "-"), ("Billing", "Private", "--")]:
    g = pd.DataFrame({"issue_type": issue, "segment": segment, "agent_experience_months": months})
    plt.plot(months, model_b.predict(g), style, label=f"{issue}, {segment}")
plt.xlabel("Agent's months in the job")
plt.ylabel("Predicted probability of escalation")
plt.legend()
plt.show()

### When to have a specialist ready

A specialist can be booked in advance when escalation is likely. Where to set the threshold is a business choice:
a lower threshold finds more of the escalations, but books a specialist for many contacts that do not need one.

In [ ]:
contacts["p_escalated"] = model_b.predict(contacts)
n_escalated = contacts["escalated"].sum()

for threshold in [0.15, 0.20, 0.25, 0.30]:
    flagged = contacts[contacts["p_escalated"] > threshold]
    found = flagged["escalated"].sum()
    print(f"Above {threshold:.2f}: {len(flagged):>4} flagged, {found:>3} escalated among them, "
          f"{found / n_escalated:.0%} of all escalations, {found / len(flagged):.0%} of the flagged")

### Exercise 2

If the threshold goes from 0.20 to 0.30, how many contacts are flagged, and what share of the escalations do you
find? Write your prediction, run the cell, then change `threshold` to 0.30 and run it again. Which would you suggest
to the center, and what would you need to know to decide?

In [ ]:
threshold = 0.20      # change to 0.30

flagged = 0
found = 0
for p, escalated in zip(contacts["p_escalated"], contacts["escalated"]):
    if p > threshold:
        flagged += 1
        found += escalated

print(f"Threshold {threshold:.2f}: {flagged} contacts flagged, {found} of them escalated")
print(f"Share of all {n_escalated} escalations found: {found / n_escalated:.0%}")

## Part 5. What the results can say

**What the model can say:** which contacts are escalated more often, at the same issue type, segment and experience;
how likely a contact we describe is to be escalated; where a threshold would put us; and how certain the estimates
are.

**What it cannot say:** that more experience causes fewer escalations. Agents are not assigned to contacts at random,
and the experienced ones may get other contacts. To learn what a change causes, the center could assign it by lot,
for example the course. Write the results as associations: "contacts handled by agents with more experience are
escalated less often".

**A summary for management.** About 13% of contacts are escalated. Technical issues are escalated far more often
than billing issues (about 15 percentage points more on average), and private customers' issues less often than
business customers' (about 7 points less). Contacts handled by more experienced agents are escalated less often,
about 2 points less per extra year in the job on average. All three are clearly significant (p 0.001 or lower;
for technical issues the 95% interval is 10 to 19 points). These are associations in the center's own data; they do
not show what would happen if agents were assigned differently. If a specialist is booked when the predicted
probability is above 20%, about 435 of 2,000 contacts would be flagged and 45% of escalations caught.

The average marginal effects behind these numbers:

In [ ]:
ame = model_b.get_margeff(at="overall", dummy=True)
print(ame.summary())

## Danish terms

| English | Danish |
|---|---|
| dependent variable | afhængig variabel |
| explanatory variable | forklarende variabel |
| control variable | kontrolvariabel |
| earlier value | tidligere værdi, udgangsniveau |
| same rows | samme observationer |
| customer service center | kundeservicecenter |
| second-line team | 2. linje |
| escalate | eskalere, sende videre |
| wrap-up | efterbehandling |